In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import PowerTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import ParameterGrid, KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.tree import plot_tree
import plotly.express as px
import matplotlib.pyplot as plt
import pickle
import joblib
from google.colab import files

# <font color='green'>Import data</font>

In [ ]:
with open('X_NO3_top25_and_y.pkl', 'rb') as f:
    data = pickle.load(f)
X = data['X']
y = data['y']

In [ ]:
with open('X_NO3_top25_and_y_trans.pkl', 'rb') as f:
    data = pickle.load(f)
X_trans = data['X']
y_trans = data['y']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#X_train_trans, X_test_trans, y_train_ttrans, y_test_trans = train_test_split(X_trans, y_trans, test_size=0.2, random_state=42)

# <font color='green'>Random Forest Regression</font>

In [ ]:
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [2, 3],
    'max_features': [None, 'sqrt', 'log2'],
    'max_leaf_nodes': [None, 10, 20],
    'bootstrap': [True],
    'min_impurity_decrease': [0.0, 0.025, 0.05],

}

In [ ]:
rf = RandomForestRegressor(random_state = 42)

grid_search = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv =  5,
    scoring = 'neg_mean_squared_error',
    n_jobs = -1,
    verbose = 2
)

grid_search.fit(X_train, y_train)

# Save the best estimator (model with best parameters)
joblib.dump(grid_search.best_estimator_, 'NO3_RF_best_model.pkl')

print("Best Parameters:", grid_search.best_params_)
print("Best Score (neg MSE)", grid_search.best_score_)



In [ ]:
#train on complete training data set using best parameters

rf_regr = RandomForestRegressor(bootstrap = True,
                                max_depth = 5,
                                max_features = 'log2',
                                max_leaf_nodes = None,
                                min_impurity_decrease = 0.0,
                                min_samples_leaf = 3,
                                min_samples_split = 2,
                                n_estimators = 150,
                                random_state = 42)

rf_regr.fit(X_train, y_train)

In [ ]:
y_pred = rf_regr.predict(X_test)

In [ ]:
print("Test RMSE:", root_mean_squared_error(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

In [ ]:
len(y_test)

In [ ]:
y_plot = pd.DataFrame({'True Values': y_test, 'Predicted Values': y_pred})
fig = px.scatter(y_plot, x = 'True Values', y = 'Predicted Values', trendline="ols")

fig.show()

In [ ]:
y_plot2 = pd.DataFrame({
    'Index': range(len(y_test)),
    'True values': y_test,
    'Predicted values': y_pred
})

df_long = y_plot2.melt(id_vars='Index', value_vars=['True values', 'Predicted values'],
                       var_name='Type', value_name='Value')

fig = px.scatter(df_long, x='Index', y='Value', color='Type',
                 title='Random Forest True vs Predicted Nitrate Values')
fig.show()

In [ ]:
residuals = y_test - y_pred

no3_res = pd.DataFrame({
    "Fitted": y_pred,
    "Residuals": residuals
})

fig = px.scatter(no3_res, x="Fitted", y="Residuals",title="Random Forest Residuals vs. Fitted Nitrate Values"
)

fig.add_hline(y=0)
fig.show()

In [ ]:
pd.DataFrame({'y_test': y_test, 'y_pred_RF': y_pred}).to_csv('Random_Forest_Predictions_NO3.csv', index = False)

# <font color='green'>Gradient Boosting Regression</font>

In [ ]:
param_grid = {
    'loss': ['squared_error'],
    'learning_rate' : [0.05, 0.1, 0.2],
    'n_estimators': [100, 150],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [2, 3],
    'max_features': [None, 'sqrt', 'log2'],
    'max_leaf_nodes': [None, 10, 20],
    'min_impurity_decrease': [0.0, 0.025, 0.05],

}

In [ ]:
gbr = GradientBoostingRegressor(random_state = 42)

grid_search_gbr = GridSearchCV(
    estimator = gbr,
    param_grid = param_grid,
    cv =  5,
    scoring = 'neg_mean_squared_error',
    n_jobs = -1,
    verbose = 2
)

grid_search_gbr.fit(X_train, y_train)

joblib.dump(grid_search_gbr.best_estimator_, 'NO3_GBR_best_model.pkl')

print("Best Parameters:", grid_search_gbr.best_params_)
print("Best Score (neg MSE)", grid_search_gbr.best_score_)

In [ ]:
#train on complete training data set using best parameters

gbr = GradientBoostingRegressor(random_state = 42,
                                   learning_rate = 0.2,
                                   loss = 'squared_error',
                                   max_depth = 10,
                                   max_features = None,
                                   max_leaf_nodes = 10,
                                   min_impurity_decrease = 0.05,
                                   min_samples_leaf = 3,
                                   min_samples_split = 2,
                                   n_estimators = 100)
gbr.fit(X_train, y_train)


In [ ]:
y_pred_gbr = gbr.predict(X_test)

print("Test RMSE:", root_mean_squared_error(y_test, y_pred_gbr))
print("Test MAE:", mean_absolute_error(y_test, y_pred_gbr))

In [ ]:
y_plot = pd.DataFrame({'True Values': y_test, 'Predicted Values': y_pred})
fig = px.scatter(y_plot, x = 'True Values', y = 'Predicted Values', trendline = 'ols')
fig.show()

In [ ]:
y_RFE_plot2 = pd.DataFrame({
    'Index': range(len(y_test)),
    'True values': y_test,
    'Predicted values': y_pred
})

df_long = y_plot2.melt(id_vars='Index', value_vars=['True values', 'Predicted values'],
                       var_name='Type', value_name='Value')

fig = px.scatter(df_long, x='Index', y='Value', color='Type',
                 title='True vs Predicted Values')
fig.show()

In [ ]:
import joblib

# Save the best estimator (model with best parameters)
joblib.dump(grid_search_gbr.best_estimator_, 'NO3_GBR_best_model.pkl')

In [ ]:
pd.DataFrame({'y_test': y_test, 'y_pred_GBR': y_pred_gbr}).to_csv('NO3_Gradient_Boosted_Regression_Predictions.csv', index = False)